In [1]:
import sqlite3
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import os

# Connect to database
db_path = r"C:\Users\tania\OneDrive\Documents\2026\2026 Tech Projects\sql-analytics-portfolio\food_economy_analysis\FoodPrices.db"
conn = sqlite3.connect(db_path)

year_cols = ['Y2015','Y2016','Y2017','Y2018','Y2019',
             'Y2020','Y2021','Y2022','Y2023']

# Load PPI data
df = pd.read_sql_query("""
    SELECT Area, Item, Element,
           Y2015, Y2016, Y2017, Y2018, Y2019,
           Y2020, Y2021, Y2022, Y2023
    FROM food_prices
    WHERE Element = 'Producer Price Index (2014-2016 = 100)'
    AND Area != ''
    AND Item != ''
""", conn)

# Clean year columns
df[year_cols] = df[year_cols].replace('', None)
df[year_cols] = df[year_cols].astype(float)

print(f"✅ Loaded {len(df):,} rows")

✅ Loaded 13,268 rows


In [3]:
# Volatility = average absolute year-on-year change
def avg_yoy_volatility(row):
    vals = [row[c] for c in year_cols if pd.notna(row[c])]
    if len(vals) < 2:
        return None
    changes = [abs(vals[i] - vals[i-1]) for i in range(1, len(vals))]
    return sum(changes) / len(changes)

df['volatility'] = df.apply(avg_yoy_volatility, axis=1)

country_vol = df.groupby('Area')['volatility'].mean().dropna()
country_vol = country_vol.sort_values(ascending=False).head(20)

fig1 = px.bar(
    country_vol,
    x=country_vol.values,
    y=country_vol.index,
    orientation='h',
    title='Top 20 Most Volatile Countries: Avg Year-on-Year PPI Change',
    labels={'x': 'Average Volatility Score', 'y': 'Country', 'color': 'Volatility Score'},
    color=country_vol.values,
    color_continuous_scale='Reds',
    color_continuous_midpoint=country_vol.mean()
)
fig1.update_layout(
    yaxis={'categoryorder': 'total ascending'},
    height=700,
    coloraxis_colorbar=dict(title='Volatility Score')
)
fig1.show()

🇮🇷 Iran is the most volatile — hyperinflation driven by sanctions
🇸🇷 Suriname appears again — currency crisis confirmed
🇹🇷 Türkiye — severe inflation crisis in 2021-2023
🇺🇦 Ukraine appearing — war impact on food prices
🇾🇪 Yemen — ongoing conflict driving instability

In [5]:
item_vol = df.groupby('Item')['volatility'].mean().dropna()
item_vol = item_vol.sort_values(ascending=False).head(20)

fig2 = px.bar(
    item_vol,
    x=item_vol.values,
    y=item_vol.index,
    orientation='h',
    title='Top 20 Most Volatile Food Items Globally',
    labels={'x': 'Average Volatility Score', 'y': 'Food Item'},
    color=item_vol.values,
    color_continuous_scale='Oranges'
)
fig2.update_layout(
    yaxis={'categoryorder': 'total ascending'},
    height=700,
    coloraxis_colorbar=dict(title='Volatility Score')
)
fig2.show()

🫘 Locust beans (carobs) is a surprise leader — niche crop with very limited global supply making it price sensitive
🍒 Sour cherries & Kiwi fruit — perishable fruits are consistently volatile
🍊 Oranges & Onions — appear in both South Africa AND globally, confirming these are genuinely volatile commodities
🍈 Melons, Dates, Watermelons — mostly warm climate fruits sensitive to weather and water supply

In [7]:
# Average PPI per country pre and post COVID
covid = df.groupby('Area')[['Y2019', 'Y2021']].mean().dropna()
covid['covid_impact'] = covid['Y2021'] - covid['Y2019']
covid = covid.sort_values('covid_impact', ascending=False).head(20)

fig3 = px.bar(
    covid,
    x='covid_impact',
    y=covid.index,
    orientation='h',
    title='COVID Impact by Country: PPI Change (2019 → 2021)',
    labels={'covid_impact': 'PPI Change (index points)', 'y': 'Country'},
    color='covid_impact',
    color_continuous_scale='Reds'
)
fig3.update_layout(
    yaxis={'categoryorder': 'total ascending'},
    height=700,
    coloraxis_colorbar=dict(title='PPI Change')
)
fig3.add_vline(x=0, line_dash='dash', line_color='grey')
fig3.show()

🇸🇷 Suriname & 🇮🇷 Iran dominate again — their food price crises were already underway before COVID and accelerated through it
🇺🇦 Ukraine appears even before the 2022 war — COVID already stressed their food system
🇾🇪 Yemen — conflict plus COVID created a devastating combination
🇧🇷 Brazil — COVID hit their agricultural supply chains hard

In [9]:
def risk_rating(score):
    if score > 200:
        return 'Critical Risk'
    elif score > 100:
        return 'High Risk'
    elif score > 50:
        return 'Elevated Risk'
    else:
        return 'Lower Risk'

risk['risk_rating'] = risk['sustained_rise'].apply(risk_rating)
risk = risk.sort_values('sustained_rise', ascending=False).head(30)

fig4 = px.bar(
    risk,
    x='sustained_rise',
    y=risk.index,
    orientation='h',
    title='Food Security Risk by Country: Sustained Price Rise (2019-2023)',
    labels={'sustained_rise': 'Sustained PPI Rise', 'y': 'Country'},
    color='risk_rating',
    color_discrete_map={
        'Critical Risk': '#d62728',
        'High Risk': '#ff4500',
        'Elevated Risk': '#ff7f0e',
        'Lower Risk': '#2ca02c'
    }
)
fig4.update_layout(
    yaxis={'categoryorder': 'total ascending'},
    height=800,
    legend_title='Risk Rating'
)
fig4.show()

🔴 Critical Risk (200+ points) — Türkiye, Suriname, Iran, Egypt, Argentina, Yemen, Guyana — all experiencing severe economic crises, conflict, or sanctions on top of global food inflation
🟠 High Risk (100-200 points) — Sri Lanka, Ukraine, Malawi, Zambia, Guinea, Mongolia, Colombia, Ethiopia, Syria, Rwanda, Belarus, Jamaica, Kazakhstan, Samoa, Nicaragua — mix of conflict, economic instability and climate vulnerability
🟡 Elevated Risk (50-100 points) — Hungary, Turkmenistan, Brazil, Ghana, Azerbaijan, Honduras, Cook Islands — affected by global inflation but more stable economies
🟢 Lower Risk — Only Kyrgyzstan in the bottom 30 — remarkably stable relative to peers

In [10]:
save_path = r"C:\Users\tania\OneDrive\Documents\2026\2026 Tech Projects\sql-analytics-portfolio\food_economy_analysis\visualisations\charts"

os.makedirs(save_path, exist_ok=True)

fig1.write_html(os.path.join(save_path, '07_country_volatility.html'))
fig2.write_html(os.path.join(save_path, '08_item_volatility.html'))
fig3.write_html(os.path.join(save_path, '09_covid_impact.html'))
fig4.write_html(os.path.join(save_path, '10_food_security_risk.html'))

print("✅ All charts saved!")

✅ All charts saved!
